1\. Write a function that converts number representation, bin<->dec<->hex. (Clearly using the corresponding python built-in functions is not fair..)

In [1]:
def dec2bin(dec: int):
    if dec == 0:
        return '0b0'
    r2 = []
    while dec > 0:
        r2.append(str(dec % 2))
        #print(r2)
        dec //= 2
    return '0b'+''.join(list(reversed(r2)))
dec2bin(100)


'0b1100100'

In [2]:
def bin2dec(bin: str):
    binary = bin[2:]
    return sum(int(b)*(2**(i)) for i, b in enumerate(reversed(binary)))

bin2dec('0b110')

6

In [3]:
def dec2hex(dec: int):
    if dec == 0:
        return '0x0'
    r16 = []
    hex_chars = '0123456789ABCDEF'
    while dec > 0:
        r16.append(hex_chars[dec%16])
        dec //= 16
    
    return '0x'+''.join(list(reversed(r16)))

def hex2dec(hex: str):
    hex_chars = '0123456789ABCDEF'
    return sum((16**i)*hex_chars.index(h.upper()) for i, h in enumerate(reversed(hex[2:])))

dec2hex(20)

'0x14'

In [4]:
#       Binary to hexadecimal

def bin2hex(bin: str):
    return dec2hex(bin2dec(bin))

def hex2bin(hex: str):
    return dec2bin(hex2dec(hex))

print(bin2hex('0b1100100'), hex2bin('0xa2d2f8c'))

0x64 0b1010001011010010111110001100


In [5]:
def number_rep_converter(value: list, from_base: str, to_base: str):
    if from_base == 'binary':
        if to_base == 'decimal':
            return bin2dec(value)
        if to_base == 'hexadecimal':
            return bin2hex(value)
    elif from_base == 'decimal':
        if to_base == 'binary':
            return dec2bin(value)
        if to_base == 'hexadecimal':
            return dec2hex(value)
    elif from_base == 'hexadecimal':
        if to_base == 'decimal':
            return hex2dec(value)
        if to_base == 'binary':
            return hex2bin(value)

number_rep_converter(value='0b1100100', from_base='binary', to_base='decimal')

100

2\. Write a function that converts a 32 bit word into a single precision floating point (i.e. interprets the various bits as sign, mantissa and exponent)

In [6]:
def f_point_converter(word: str, base: str, bias: int = 127):
    binary = list(hex2bin(word))[2:] if base=='hexadecimal' else list(word)[2:]
    while len(binary) < 32:
        binary.append('0')
    print(binary)
    binary = list(reversed(binary))
    print(len(binary))
    s = int(binary[31])
    print(s)
    e = int(bin2dec(list(reversed(binary[23:30]))[2:]))
    f = binary[:22]
    print(f)
    f_point = float(f"1.{1+sum(int(m)**i for i, m in enumerate(f))}")
    return ((-1)**s)*f_point*(2**(e-bias))

f_point_converter(word='0b00001010001011010010111110001100', base='binary')

['0', '0', '0', '0', '1', '0', '1', '0', '0', '0', '1', '0', '1', '1', '0', '1', '0', '0', '1', '0', '1', '1', '1', '1', '1', '0', '0', '0', '1', '1', '0', '0']
32
0
['0', '0', '1', '1', '0', '0', '0', '1', '1', '1', '1', '1', '0', '1', '0', '0', '1', '0', '1', '1', '0', '1']


1.0720508479499261e-37

3\. Write a program to determine the underflow and overflow limits (within a factor of 2) for python on your computer. 

**Tips**: define two variables inizialized to 1 and halve/double them enough time to exceed the under/over-flow limits  

In [7]:
#   Program that repeatedly halves (underflow) or double (overflow) a f-point number
#   until it is no longer representable

def fpoint_limit():
    underflow, overflow = 1.0, 1.0
    while underflow / 2 > 0:    # different result if I divide here by 2 or not
        underflow /= 2
    while overflow * 2 < float('inf'):  # different result if I multiply here by 2 or not
        overflow *= 2
    return underflow, overflow

underflow_lim, overflow_lim = fpoint_limit()

print("underflow limit:", underflow_lim, "\n",
      "overflow limit:", overflow_lim)

underflow limit: 5e-324 
 overflow limit: 8.98846567431158e+307


4\. Write a program to determine the machine precision

**Tips**: define a new variable by adding a smaller and smaller value (proceeding similarly to prob. 2) to an original variable and check the point where the two are the same 

In [8]:
# We want to find the smallest number epsilon such that when added to a number
# the result is still distinguishable in f-point arithmetic

def machprec():
    x = 1
    e = 1.0
    while x + e > x:
        e /= 2
    return e

print(f"Machine precision up to (epsilon): {machprec():.16f}")

Machine precision up to (epsilon): 0.0000000000000001


5\. Write a function that takes in input three parameters $a$, $b$ and $c$ and prints out the two solutions to the quadratic equation $ax^2+bx+c=0$ using the standard formula:
$$
x=\frac{-b\pm\sqrt{b^2-4ac}}{2a}
$$

(a) use the program to compute the solution for $a=0.001$, $b=1000$ and $c=0.001$

(b) re-express the standard solution formula by multiplying top and bottom by $-b\mp\sqrt{b^2-4ac}$ and again find the solution for $a=0.001$, $b=1000$ and $c=0.001$. How does it compare with what previously obtained? Why?

(c) write a function that compute the roots of a quadratic equation accurately in all cases

In [22]:
import math
import numpy as np
def roots(a, b, c):
    root1 = (-b + math.sqrt(b**2 - 4*a*c)) / (2 * a)
    root2 = (-b - math.sqrt(b**2 - 4*a*c)) / (2 * a)
    return root1, root2

a=0.1
b=10000000
c=0.1

In [23]:
# a)
roots(a, b, c)

(-9.313225746154785e-09, -100000000.0)

In [24]:
# b)
def rootsB(a, b, c):
    root1 = (2*c) / (-b -math.sqrt(b**2 - 4*a*c))
    root2 = (2*c) / (-b +math.sqrt(b**2 - 4*a*c))
    return root1, root2

rootsB(a, b, c)

(-1e-08, -107374182.4)

In [25]:
def rootsC(a, b, c):
    root1 = (2*c) / (-b -math.sqrt(b**2 - 4*a*c))
    root2 = (-b - math.sqrt(b**2 - 4*a*c)) / (2 * a)
    return root1, root2

rootsB(a, b, c)

(-1e-08, -107374182.4)

6\. Write a program that implements the function $f(x)=x(x−1)$

(a) Calculate the derivative of the function at the point $x = 1$ using the derivative definition:

$$
\frac{{\rm d}f}{{\rm d}x} = \lim_{\delta\to0} \frac{f(x+\delta)-f(x)}{\delta}
$$

with $\delta = 10^{−2}$. Calculate the true value of the same derivative analytically and compare with the answer your program gives. The two will not agree perfectly. Why not?

(b) Repeat the calculation for $\delta = 10^{−4}, 10^{−6}, 10^{−8}, 10^{−10}, 10^{−12}$ and $10^{−14}$. How does the accuracy scales with $\delta$?

In [24]:
def f(x):
    return x * (x - 1)

def f_prime_analytical(x):
    return 2*x - 1    

def f_prime_numerical(x, delta):
    return ( f(x+delta) - f(x) ) / delta
print("Analytical Result\tNumerical Result")
print("-"*50)
print(f"{f_prime_analytical(1)}\t             |  {f_prime_numerical(1, 1e-2)}")

Analytical Result	Numerical Result
--------------------------------------------------
1	             |  1.010000000000001


In [30]:
deltas = [1e-2, 1e-4, 1e-6, 1e-8, 1e-10, 1e-12, 1e-14, 1e-16]
x0 = 1
analytical_result = f_prime_analytical(x0)

print(f"Analytical Derivative at x={x0}: {analytical_result:.10f}\n")

print("δ\tNumerical Derivative\tAbsolute Error")
print("-" * 50)

for delta in deltas:
    numerical_result = f_prime_numerical(x0, delta)
    error = abs(analytical_result - numerical_result)
    print(f"{delta:.0e}\t{numerical_result:.10f}\t        {error:.10f}")

Analytical Derivative at x=1: 1.0000000000

δ	Numerical Derivative	Absolute Error
--------------------------------------------------
1e-02	1.0100000000	        0.0100000000
1e-04	1.0001000000	        0.0001000000
1e-06	1.0000009999	        0.0000009999
1e-08	1.0000000039	        0.0000000039
1e-10	1.0000000828	        0.0000000828
1e-12	1.0000889006	        0.0000889006
1e-14	0.9992007222	        0.0007992778
1e-16	0.0000000000	        1.0000000000


###### Absolute Error reaches a minimum at delta = 1e-8 and then increases due to floating point precision limit as numerical errors amplify due to incorrect rounding of f(x+delta) - f(x) for small delta

7\. Consider the integral of the semicircle of radius 1:
$$
I=\int_{-1}^{1} \sqrt(1-x^2) {\rm d}x
$$
which it's known to be $I=\frac{\pi}{2}=1.57079632679...$.
Alternatively we can use the Riemann definition of the integral:
$$
I=\lim_{N\to\infty} \sum_{k=1}^{N} h y_k 
$$

with $h=2/N$ the width of each of the $N$ slices the domain is divided into and where
$y_k$ is the value of the function at the $k$-th slice.

(a) Write a programe to compute the integral with $N=100$. How does the result compares to the true value?

(b) How much can $N$ be increased if the computation needs to be run in less than a second? What is the gain in running it for 1 minute?

In [ ]:
import numpy as np
def semicircle_y(x):
    return np.sqrt( 1 - x**2)

N = 100
h = 2 / N
k = np.linspace(-1 + h/2, 1 - h/2, N)
riemann = [semicircle_y(_) for _ in k ] 
I = h * np.sum(riemann)

print(f" Numerical result: {I}", "\n",
      f"True Value:       {np.pi/2}", "\n",
      f"Absolute Error:   {abs(I - np.pi/2):.0e}"
)

 Numerical result: 1.571282776229796 
 True Value:       1.5707963267948966 
 Absolute Error:   5e-04


In [ ]:
import time

N_values = [ 10**i for i in range(1, 9) ]
for i, N in enumerate(N_values):
    start_time = time.time()
    I = 2 / N * np.sum([ semicircle_y(_) for _ in np.linspace(-1 + 1/N, 1 - 1/N, N) ])
    elapsed_time = time.time() - start_time
    print("N\tAbsolute Error\tElapsed time (seconds)")
    print(f"{N_values[i]:.0e}\t{abs(np.pi/2 - I):.6f}\t{elapsed_time:.0e}")

N	Absolute Error	Elapsed time (seconds)
1e+01	0.015198	1e-04
N	Absolute Error	Elapsed time (seconds)
1e+02	0.000486	1e-04
N	Absolute Error	Elapsed time (seconds)
1e+03	0.000015	9e-04
N	Absolute Error	Elapsed time (seconds)
1e+04	0.000000	8e-03
N	Absolute Error	Elapsed time (seconds)
1e+05	0.000000	6e-02
N	Absolute Error	Elapsed time (seconds)
1e+06	0.000000	5e-01
N	Absolute Error	Elapsed time (seconds)
1e+07	0.000000	5e+00
N	Absolute Error	Elapsed time (seconds)
1e+08	0.000000	5e+01
